In [16]:
!pip install geopandas
!pip install flexpolyline
import geopandas as gpd
import requests
from shapely.geometry import shape
from requests.adapters import HTTPAdapter
import time
import pandas as pd
#from shapely.ops import cascaded_union,unary_union
from collections import Counter
import flexpolyline as fp
from shapely import geometry
import re
!pip install tqdm
import tqdm
from tqdm.auto import tqdm
from matplotlib import pyplot as plt
from shapely.geometry import Polygon
import  geopandas  as  gpd
from shapely import wkt
pd.set_option('display.max_columns', None)


#Function to get  BQ data as dataframe
def get_bq_table(query):

    client = bigquery.Client(project=project_id)

    df = client.query(
      query
    ).to_dataframe()
    return df

class RetryHTTPAdapter(HTTPAdapter):

  SECONDS_BETWEEN_RETRIES = 5

  def __init__(self, retry_time=75, *args, **kwargs):
    self.retry_time = retry_time
    super(RetryHTTPAdapter, self).__init__(*args, **kwargs)

  def send(self, *args, **kwargs):
    for _ in range(int(self.retry_time / self.SECONDS_BETWEEN_RETRIES)):
          response = super(RetryHTTPAdapter, self).send(*args, **kwargs)
          print(response.status_code)
          if response.status_code in [200, 204, 400, 401, 403, 404, 500]:
            break
          time.sleep(self.SECONDS_BETWEEN_RETRIES)
    return response

def new_session():
    s = requests.Session()
    s.mount('https://', RetryHTTPAdapter(retry_time=75))
    return s
def try_or(func, default=None, expected_exc=(Exception,)):
    try:
        return func()
    except expected_exc:
        return default

apikey='oEKIZLvTTjT9naP2OjzHPk6hhNO3M9dxD9vYLbD-wy4'
apikey='9wZ1Eq_mXHmNDXaE0sWLh6OV8Tjpgu_3tkWAB1M6tiU'
apikey='e5R9pRgxW2lanNcUfxWU-KeMB70r7tY3e_q0fD7ZF-U'
api_key='ELzRUipw3eGEI3WELFTUs3i-GET-fTX6OtXJYsB6KZo'
session = new_session()


In [10]:
data_df=pd.read_csv(r'C:\Users\darey\OneDrive\Documents\Quartile 1\Fundamentals of Spatial Data Engineering\Assignments\Assignment - Fundamentals of Spatial Data Engineering\Kenya_HC_Facilities.csv')
data_df.head()

,gpkg_fid,gml_id,FID,Facility_N,F_NAME,HMIS,Province,District,Division,LOCATION,Sub_Locati,Spatial_Re,Facility_T,Agency,Latitude,Longitude,GlobalID
0,1,facilities.1064,1064,1,ABIDHA HC,1343,NYANZA,BONDO,RARIEDA,EAST ASEMBO,OMIA MWALO,GPS,3,MOH,-0.15902,34.41064,{790A950A-9DF5-43E6-BE28-0C2719BABC13}
1,2,facilities.1067,1067,2,ANYUONGI DISP,0,NYANZA,BONDO,BONDO,SOUTH SAKWA,EAST MIGWENA,GPS,4,MOH,-0.17618,34.27613,{178B56FF-4E2C-4CC4-B008-701B53B35321}
2,5,facilities.1079,1079,5,GOBEI DISP,2467,NYANZA,BONDO,BONDO,BONDO TOWNSHIP,NYAWITA(BONDO),GPS,4,MOH,-0.09021,34.29050,{22E91078-025D-47DF-9EF4-E79A4190FA30}
3,6,facilities.1085,1085,6,GOT AGULU HEALTH CENTRE,1346,NYANZA,BONDO,USIGU,WEST YIMBO,GOT AGULU,GPS,3,MOH,-0.03410,34.02480,{4F23AD9B-8747-48B9-BD48-A1F98ED12770}
4,7,facilities.1089,1089,7,GOT MATAR DISP,1347,NYANZA,BONDO,USIGU,NORTH YIMBO,NYAMONYE,GPS,4,MOH,-0.03090,34.13150,{5E43EAD4-5B92-4DAF-8C25-1188C5A1038B}


In [17]:

def hereapi(df, latitude, longitude, time):
    time = int(time * 60)  # minutes -> seconds
    gdf = gpd.GeoDataFrame()
    for i in tqdm(range(len(df))):
        try:
            url = 'https://isoline.router.hereapi.com/v8/isolines'
            headers = {'Content-Type': 'application/json'}
            my_params = {
                'transportMode': 'pedestrian',           # <-- walking
                'range[type]': 'time',
                'range[values]': str(time),
                'origin': f"{df.iloc[i][latitude]},{df.iloc[i][longitude]}",
                'apikey': apikey
            }

            response = session.get(url, params=my_params, headers=headers)
            if response.status_code == 200:
                data = response.json()
                poly = fp.decode(data['isolines'][0]['polygons'][0]['outer'])
                poly = [(p[1], p[0]) for p in poly]
                geom = geometry.Polygon(poly)
                gdf = pd.concat([gdf, gpd.GeoDataFrame(geometry=[geom])])
        except Exception as e:
            print(e)
            # fall back to keeping alignment even if a row fails
            gdf = pd.concat([gdf, gpd.GeoDataFrame(geometry=[None])])

    return gpd.GeoDataFrame(df.reset_index(), geometry=gdf.reset_index()['geometry'])

In [26]:
apikey='oEKIZLvTTjT9naP2OjzHPk6hhNO3M9dxD9vYLbD-wy4'
#apikey='9wZ1Eq_mXHmNDXaE0sWLh6OV8Tjpgu_3tkWAB1M6tiU'
#apikey='e5R9pRgxW2lanNcUfxWU-KeMB70r7tY3e_q0fD7ZF-U'
#api_key='ELzRUipw3eGEI3WELFTUs3i-GET-fTX6OtXJYsB6KZo'
data1_here2=hereapi(data_df
                      ,'Latitude'
                      ,'Longitude'
                      ,30)

  0%|          | 4/1013 [00:00<01:24, 11.98it/s]

200
200
200
200
200


  1%|          | 9/1013 [00:00<01:07, 14.77it/s]

200
200
200
200


  1%|          | 11/1013 [00:00<01:14, 13.45it/s]

...

200
200
200
200
200


100%|█████████▉| 1009/1013 [01:07<00:00, 16.06it/s]

200
200
200


100%|██████████| 1013/1013 [01:08<00:00, 14.89it/s]

200
200
200


In [27]:
data1_here2['walking_time']=30
data1_here2.to_file(r'C:\Users\darey\OneDrive\Documents\Quartile 1\Fundamentals of Spatial Data Engineering\Assignments\Assignment - Fundamentals of Spatial Data Engineering\Kenya_HC_Facilities_Walking_30min.geojson', driver='GeoJSON') 

c:\Users\darey\.conda\envs\GIS\Lib\site-packages\pyogrio\geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


In [30]:
dat30=gpd.read_file(r'C:\Users\darey\OneDrive\Documents\Quartile 1\Fundamentals of Spatial Data Engineering\Assignments\Assignment - Fundamentals of Spatial Data Engineering\Kenya_HC_Facilities_Walking_30min.geojson')
dat60=gpd.read_file(r'C:\Users\darey\OneDrive\Documents\Quartile 1\Fundamentals of Spatial Data Engineering\Assignments\Assignment - Fundamentals of Spatial Data Engineering\Kenya_HC_Facilities_Walking_60min.geojson')
pd.concat([dat30,dat60]).to_file(r'C:\Users\darey\OneDrive\Documents\Quartile 1\Fundamentals of Spatial Data Engineering\Assignments\Assignment - Fundamentals of Spatial Data Engineering\Kenya_HC_Facilities_WalkingTime.geojson', driver='GeoJSON')

In [ ]:
data1_here2.to_file('drivetimes2.geojson')

/usr/local/lib/python3.11/dist-packages/pyogrio/geopandas.py:662: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


In [ ]:
data1_here.to_csv('carwash5mins.csv')

In [ ]:
from google.cloud import bigquery
import pandas as pd
import pandas_gbq  # Correct impor

# Set up the BigQuery client
client = bigquery.Client()


pandas_gbq.to_gbq(data1_here,destination_table='soleng-dev-services.ExpresscCarWash.drivetimes5',
    project_id='soleng-dev-services',
    if_exists="replace")

100%|██████████| 1/1 [00:00<00:00, 505.16it/s]


In [ ]:
to_bq(data1_here,destination_table='soleng-dev-services.ExpresscCarWash.drivetimes5',
    project_id='soleng-dev-services',
    if_exists="replace")

NameError: name 'to_bq' is not defined

In [ ]:
api_key

In [ ]:
df.to_csv('no.csv')

In [ ]:
df=data_df.copy()
latitude='Latitud'
longitude='Longitud'
time_=int(10*60)

gdf=gpd.GeoDataFrame()
for i in tqdm(range(len(df))):
    try:


        url='https://isoline.router.hereapi.com/v8/isolines'
        headers = {'Content-Type': 'application/json'}
        my_params = {'transportMode':'car'
                      ,'range[type]':'time'
                      ,'range[values]':str(time_)
                      ,'origin':str(df.iloc[i][latitude])+','+str(df.iloc[i][longitude])
                      ,'apikey':'ELzRUipw3eGEI3WELFTUs3i-GET-fTX6OtXJYsB6KZo'}

        response = session.get(url, params=my_params ,headers=headers)

        if response.status_code==200:
            data=response.json()
            poly=fp.decode(data['isolines'][0]['polygons'][0]['outer'])
            poly=[ (i[1],i[0]) for i in poly]
            geom=geometry.Polygon(poly)

            gdf=pd.concat([gdf,gpd.GeoDataFrame(geometry=[geom])])
        else:
            pass
            #gdf=pd.concat([gdf,gpd.GeoDataFrame(geometry=[geom])])
    except Exception as e:
        print(e)
        gdf=pd.concat([gdf,gpd.GeoDataFrame(geometry=[geom])])

In [ ]:
dat=gpd.GeoDataFrame(df.reset_index(),geometry=gdf.reset_index()['geometry'])

In [ ]:
dat

In [ ]:
dat[dat['geometry'] != None].to_csv('drive_times3.csv')

In [ ]:
pd.concat([gdf,gpd.GeoDataFrame(geometry=[geom])])

In [ ]:
response.json()

In [ ]:
data1_here=hereapi(data_df
                      ,'Latitud'
                      ,'Longitud'
                      ,10)

In [ ]:
data1_here.json()

In [ ]:
data1_here=hereapi(data_df.query("Type == 'Suburban'")
                      ,'lat'
                      ,'long'
                      ,15)

data2_here=hereapi(data_df.query("Type == 'Rural'")
                      ,'lat'
                      ,'long'
                      ,20)

data3_here=hereapi(data_df.query("Type == 'Urban'")
                      ,'lat'
                      ,'long'
                      ,10)


In [ ]:
pd.concat([data1_here,data2_here,data3_here]).to_csv('dat4.csv')

In [ ]:
data1_here[[	'NumCliente'	,'Longitud_X'	,'Latitud_X'	,'geometry']].to_csv('dat4.csv')

In [ ]:
data1_here[[	'NumCliente'	,'Longitud_X'	,'Latitud_X'	,'geometry']].to_file('drive.geojson')

In [ ]:
data1_here.to_file('10mins_vystart.geojson')
data1_here.to_csv('10mins_vystart.csv')

In [ ]:
data2=data_df.query("type == 'Urban'")
data2_here=hereapi(data2,'lat','long',5)

In [ ]:
full_dat=pd.concat([data1_here,data2_here])

In [ ]:
full_dat[full_dat['id'].isin(data_df2.id.to_list())].to_csv('sams_drivetime.csv')